# VLSP 2025 KD — Notebook 2: SFT Only (Gold Data Baseline)

**Goal**: Train student (Qwen3.5-4B) on gold SFT data (no teacher distillation)
**Baseline**: What EA/PA does SFT alone achieve?
**Output**: SFT adapter + EA/PA metrics for comparison with NB3

| Step | Time |
|------|------|
| Test mode (50 samples) | ~15-20 min |
| Full run | ~1.5-2h |

This is the **fastest** notebook — ideal to run first to establish a baseline.

## Execution Order
1. Run ALL cells with `TEST_MODE=True` (~15 min)
2. Verify Cell 6 shows loss < 0.5 and Cell 7 shows valid EA/PA
3. Set `TEST_MODE=False`, restart kernel, full run (~1.5-2h)

In [1]:
!pip install /kaggle/input/datasets/thanhduc1102/vlsp2025-kd-wheels/huggingface_hub-1.11.0-py3-none-any.whl --no-deps --force-reinstall
!pip install /kaggle/input/datasets/thanhduc1102/vlsp2025-kd-wheels/transformers-5.5.4-py3-none-any.whl --no-deps --force-reinstall

Processing /kaggle/input/datasets/thanhduc1102/vlsp2025-kd-wheels/huggingface_hub-1.11.0-py3-none-any.whl
  Attempting uninstall: huggingface-hub


    Found existing installation: huggingface_hub 1.4.1


    Uninstalling huggingface_hub-1.4.1:


      Successfully uninstalled huggingface_hub-1.4.1


Processing /kaggle/input/datasets/thanhduc1102/vlsp2025-kd-wheels/transformers-5.5.4-py3-none-any.whl


  Attempting uninstall: transformers


    Found existing installation: transformers 5.0.0


    Uninstalling transformers-5.0.0:


      Successfully uninstalled transformers-5.0.0


In [2]:
import transformers
print(f"Transformers version: {transformers.__version__}")

Transformers version: 5.5.4


In [3]:
!pip install transformers accelerate bitsandbytes peft trl datasets \
    --no-index \
    --find-links=/kaggle/input/datasets/thanhduc1102/vlsp2025-kd-wheels/ \
    -U

Looking in links: /kaggle/input/datasets/thanhduc1102/vlsp2025-kd-wheels/


Processing /kaggle/input/datasets/thanhduc1102/vlsp2025-kd-wheels/transformers-5.6.0-py3-none-any.whl


Processing /kaggle/input/datasets/thanhduc1102/vlsp2025-kd-wheels/accelerate-1.13.0-py3-none-any.whl


Processing /kaggle/input/datasets/thanhduc1102/vlsp2025-kd-wheels/bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl


Processing /kaggle/input/datasets/thanhduc1102/vlsp2025-kd-wheels/peft-0.19.1-py3-none-any.whl
Processing /kaggle/input/datasets/thanhduc1102/vlsp2025-kd-wheels/trl-1.2.0-py3-none-any.whl
Processing /kaggle/input/datasets/thanhduc1102/vlsp2025-kd-wheels/datasets-4.8.4-py3-none-any.whl


INFO: pip is looking at multiple versions of huggingface-hub to determine which version is compatible with other requirements. This could take a while.


Processing /kaggle/input/datasets/thanhduc1102/vlsp2025-kd-wheels/transformers-5.5.4-py3-none-any.whl


ERROR: Could not find a version that satisfies the requirement hf-xet<2.0.0,>=1.4.3; platform_machine == "x86_64" or platform_machine == "amd64" or platform_machine == "AMD64" or platform_machine == "arm64" or platform_machine == "aarch64" (from huggingface-hub) (from versions: none)
ERROR: No matching distribution found for hf-xet<2.0.0,>=1.4.3; platform_machine == "x86_64" or platform_machine == "amd64" or platform_machine == "AMD64" or platform_machine == "arm64" or platform_machine == "aarch64"


In [4]:
# ════════════════════════════════════════════════════════════════
# CELL 1: CONFIGURATION
# ════════════════════════════════════════════════════════════════

TEST_MODE    = False     # ← SET TO False AFTER TEST PASSES
TEST_SAMPLES = 50       # samples in test mode

NOTEBOOK_ID = "sft-only"
OUTPUT_DIR  = f"/kaggle/working/outputs/{NOTEBOOK_ID}"

import os
from pathlib import Path
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print(f"{'='*60}")
print(f"Notebook  : {NOTEBOOK_ID}  (SFT on gold data — no teacher distillation)")
print(f"TEST_MODE : {TEST_MODE}  (n={TEST_SAMPLES if TEST_MODE else 'ALL'})")
print(f"Output    : {OUTPUT_DIR}")
print(f"{'='*60}")


Notebook  : sft-only  (SFT on gold data — no teacher distillation)
TEST_MODE : False  (n=ALL)
Output    : /kaggle/working/outputs/sft-only


In [5]:
import subprocess, os, sys, shutil
from pathlib import Path

WORK_DIR     = Path("/kaggle/working/vlsp2025")
WHEELS_DIR   = Path("/kaggle/input/datasets/thanhduc1102/vlsp2025-kd-wheels")
PIPELINE_SRC = Path("/kaggle/input/datasets/thanhduc1102/vlsp2025-kd-pipeline")

os.environ.update({
    "HF_HUB_OFFLINE": "1",
    "TRANSFORMERS_OFFLINE": "1",
    "TOKENIZERS_PARALLELISM": "false",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
})
WORK_DIR.mkdir(parents=True, exist_ok=True)

# --- Install wheels ---
if WHEELS_DIR.exists():
    wheels = sorted(WHEELS_DIR.glob("*.whl"))
    if wheels:
        r = subprocess.run(
            [sys.executable, "-m", "pip", "install", "--quiet", "--no-index",
             f"--find-links={WHEELS_DIR}"] + [str(w) for w in wheels],
            capture_output=True, text=True)
        print(f"Wheels: {len(wheels)} installed" if r.returncode == 0 else f"Wheel warn: {r.stderr[:100]}")
else:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                    "transformers", "accelerate", "peft", "bitsandbytes", "sympy", "tqdm"],
                   check=False)
    print("Installed from PyPI (online mode)")

# --- Copy pipeline code (pipeline/, src/, configs/) ---
if PIPELINE_SRC.exists():
    for d in ["pipeline", "src", "configs"]:
        src_d = PIPELINE_SRC / d
        dst_d = WORK_DIR / d
        if src_d.exists():
            if dst_d.exists():
                shutil.rmtree(dst_d)
            shutil.copytree(src_d, dst_d)
            n = sum(1 for _ in dst_d.rglob("*") if _.is_file())
            print(f"  Copied {d}/ ({n} files)")
        else:
            print(f"  WARNING: {d}/ not found in pipeline dataset")
    # Verify critical module exists
    if not (WORK_DIR / "pipeline" / "__init__.py").exists():
        raise RuntimeError("pipeline/__init__.py missing after copy — re-upload vlsp2025-kd-pipeline dataset")
elif (WORK_DIR / "pipeline").exists():
    print(f"Pipeline already present: {WORK_DIR / 'pipeline'}")
else:
    raise RuntimeError(f"Pipeline not found at {PIPELINE_SRC}\nAdd dataset: thanhduc1102/vlsp2025-kd-pipeline")

if str(WORK_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_DIR))
os.chdir(WORK_DIR)
print(f"WORK_DIR: {WORK_DIR}")


Wheel warn: ERROR: pandas-3.0.2-cp313-cp313-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl is not a supported w
  Copied pipeline/ (11 files)
  Copied src/ (18 files)
  Copied configs/ (4 files)
WORK_DIR: /kaggle/working/vlsp2025


In [6]:
import sys, os, torch
from pathlib import Path

WORK_DIR = Path("/kaggle/working/vlsp2025")
if str(WORK_DIR) not in sys.path: sys.path.insert(0, str(WORK_DIR))
os.chdir(WORK_DIR)

if not torch.cuda.is_available():
    raise RuntimeError("No GPU! Enable RTX 6000 accelerator in notebook settings.")

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU:  {gpu_name}  ({vram_gb:.0f} GB)")

import transformers, peft, accelerate
print(f"Transformers: {transformers.__version__}  PEFT: {peft.__version__}")

try:
    import flash_attn
    HAS_FLASH = True
    print(f"Flash Attn: {flash_attn.__version__}")
except ImportError:
    HAS_FLASH = False
    print("Flash Attn: not available (sdpa fallback)")

GPU_PROFILE = "rtx6000_96gb" if vram_gb > 80 else ("a100_80gb" if vram_gb > 60 else "p100_16gb")
print(f"GPU Profile: {GPU_PROFILE}")

# Resolve offline model paths (Kaggle offline mode)
def resolve_model(hf_id):
    name = hf_id.split("/")[-1].lower()
    for root in [Path("/kaggle/input"), Path("/kaggle/models")]:
        if not root.exists(): continue
        for d in root.rglob("config.json"):
            parent = d.parent
            if name.replace("-","").replace("_","") in parent.name.lower().replace("-","").replace("_",""):
                print(f"  Found offline: {parent}")
                return str(parent)
    return hf_id

TEACHER_PATH = resolve_model("Qwen/Qwen3.5-27B")
STUDENT_PATH = resolve_model("Qwen/Qwen3.5-4B")
# Fallback to Kaggle model mount paths
if TEACHER_PATH == "Qwen/Qwen3.5-27B":
    p = Path("/kaggle/input/models/thanhduc1102/qwen_35_27b/transformers/default/1")
    if p.exists(): TEACHER_PATH = str(p)
if STUDENT_PATH == "Qwen/Qwen3.5-4B":
    p = Path("/kaggle/input/models/thanhduc1102/qwen_35_4b/transformers/default/1")
    if p.exists(): STUDENT_PATH = str(p)

print(f"Teacher: {TEACHER_PATH}")
print(f"Student: {STUDENT_PATH}")


GPU:  NVIDIA RTX PRO 6000 Blackwell Server Edition  (95 GB)


Transformers: 5.5.4  PEFT: 0.18.1
Flash Attn: not available (sdpa fallback)
GPU Profile: rtx6000_96gb
Teacher: /kaggle/input/models/thanhduc1102/qwen_35_27b/transformers/default/1
Student: /kaggle/input/models/thanhduc1102/qwen_35_4b/transformers/default/1


In [7]:
import sys, os
from pathlib import Path
from pipeline.config import load_config, save_config

WORK_DIR = Path("/kaggle/working/vlsp2025")
if str(WORK_DIR) not in sys.path: sys.path.insert(0, str(WORK_DIR))
os.chdir(WORK_DIR)

# ── 12h Kaggle fit strategy ──────────────────────────────────────────
# Prior run: 19s/step * 2748 steps = 14.5h  →  exceeds 12h cap.
# Root cause analysis from last session output:
#   • Flash-attn unavailable in Kaggle image → SDPA (2-3× slower)
#   • 3 epochs over-trains: loss hit 0.02 by step 2000 (memorisation)
#   • seq_len=2048 wasted on samples that fit in 1536
# Fix: shorter seq, fewer epochs, runtime watchdog, sample cap, mirror
# checkpoints to /kaggle/working/outputs/ so a session kill still leaves
# a usable adapter.
# ─────────────────────────────────────────────────────────────────────
SFT_MIRROR_DIR = f"{OUTPUT_DIR}/sft_adapter_mirror"
Path(SFT_MIRROR_DIR).mkdir(parents=True, exist_ok=True)

# Headroom below Kaggle's 12h wall. Watchdog stops training cleanly at
# this mark, triggers a save, and lets the remaining cells run (final
# save, eval, upload). Leave ~60-90 min for those.
SFT_MAX_RUNTIME_HOURS = 10.5

# Sample cap — raise if the session proves it can handle more.
# 8000 samples × 2 epochs ≈ 1000 steps at bs=1 × ga=16; at 4s/step on
# flash-attn or 10s/step on SDPA → 1.1h or 2.8h. Leaves huge margin.
SFT_MAX_TRAIN_SAMPLES = 8000

cfg = load_config(gpu_profile=GPU_PROFILE, overrides={
    "model": {"teacher_model": TEACHER_PATH, "student_model": STUDENT_PATH,
              "use_flash_attention": HAS_FLASH},
    "data": {
        "vinumqa_train":        "/kaggle/input/datasets/thanhduc1102/vinumericalqa-private/train.json",
        "vinumqa_valid":        "/kaggle/input/datasets/thanhduc1102/vinumericalqa-private/valid.json",
        "vinumqa_test":         "/kaggle/input/datasets/thanhduc1102/vinumericalqa-private/test.json",
        "vinumqa_private_test": "/kaggle/input/datasets/thanhduc1102/vinumericalqa-private/private_test.json",
        "finqa_dir":            "/kaggle/input/datasets/thanhduc1102/finqa-en",
        "max_samples":          TEST_SAMPLES if TEST_MODE else None,
    },
    "sft": {
        "num_epochs":            2,       # 3→2: loss already overfits at ep 2
        "lora_r":                64,      # smaller r for gold-only
        "max_seq_length":        1536,    # 2048→1536: most samples fit
        "per_device_train_batch_size": 1,
        "gradient_accumulation_steps": 16,
        "save_steps":            200,
        "eval_steps":            200,
        "save_total_limit":      2,
        "max_train_samples":     SFT_MAX_TRAIN_SAMPLES,
        "max_runtime_hours":     SFT_MAX_RUNTIME_HOURS,
        "mirror_save_dir":       SFT_MIRROR_DIR,
    },
    "inference": {"num_candidates": 1, "batch_size": 8},
})
print(f"Student: {cfg.model.student_model.split('/')[-1]}")
print(f"SFT: epochs={cfg.sft.num_epochs}  batch={cfg.sft.per_device_train_batch_size}x{cfg.sft.gradient_accumulation_steps}  LoRA r={cfg.sft.lora_r}  seq={cfg.sft.max_seq_length}")
print(f"SFT fit-12h: cap={cfg.sft.max_train_samples}  watchdog={cfg.sft.max_runtime_hours}h  mirror={cfg.sft.mirror_save_dir}")
save_config(cfg, str(WORK_DIR / "data/pipeline/config_sft_only.yaml"))


Student: 1
SFT: epochs=3  batch=4x4  LoRA r=64  seq=2048
Config saved: /kaggle/working/vlsp2025/data/pipeline/config_sft_only.yaml


In [8]:
import os, sys, time, json
from pathlib import Path
from pipeline.data_prep import run_data_prep

WORK_DIR = Path("/kaggle/working/vlsp2025")
if str(WORK_DIR) not in sys.path: sys.path.insert(0, str(WORK_DIR))
os.chdir(WORK_DIR)

print("=" * 60)
print("DATA PREPARATION")
print("=" * 60)
t0 = time.time()
data_paths = run_data_prep(cfg)
print(f"\nCompleted in {time.time()-t0:.1f}s")

# Print sample counts
for k, v in data_paths.items():
    try:
        d = json.load(open(v))
        print(f"  {k}: {len(d)} samples  ({v})")
    except Exception:
        print(f"  {k}: {v}")


DATA PREPARATION


ViNumQA train: 2993 samples


ViNumQA valid: 584 samples


FinQA train.json: 6251 samples
FinQA dev.json: 883 samples


FinQA test.json: 1147 samples

Total training samples (raw): 11274
Validation samples (raw): 584
Teacher mode: GUIDED (think.py) — gold program/answer embedded in prompt


Saved 11274 samples → /kaggle/working/vlsp2025/data/pipeline/teacher_input.json


Saved 14661 samples → /kaggle/working/vlsp2025/data/pipeline/sft_train.json
Saved 584 samples → /kaggle/working/vlsp2025/data/pipeline/sft_valid.json


Saved 14661 GRPO train → /kaggle/working/vlsp2025/data/pipeline/grpo_train.parquet
Saved 584 GRPO valid → /kaggle/working/vlsp2025/data/pipeline/grpo_valid.parquet

Data preparation complete!
  SFT train:  14661 samples
  SFT valid:  584 samples
  GRPO train: 14661 samples
  GRPO valid: 584 samples
  Teacher:    11274 samples


Completed in 4.9s


  teacher_input: 11274 samples  (/kaggle/working/vlsp2025/data/pipeline/teacher_input.json)


  sft_train: 14661 samples  (/kaggle/working/vlsp2025/data/pipeline/sft_train.json)
  sft_valid: 584 samples  (/kaggle/working/vlsp2025/data/pipeline/sft_valid.json)
  grpo_train: /kaggle/working/vlsp2025/data/pipeline/grpo_train.parquet
  grpo_valid: /kaggle/working/vlsp2025/data/pipeline/grpo_valid.parquet


In [9]:
# ════════════════════════════════════════════════════════════════
# CELL 6: SFT TRAINING ON GOLD DATA — 12h Kaggle fit
# Safety: runtime watchdog + periodic mirror-save to /kaggle/working/outputs
# ════════════════════════════════════════════════════════════════
import dataclasses, gc, os, sys, time, torch
from pathlib import Path
from pipeline.train_sft import run_sft_training

WORK_DIR = Path("/kaggle/working/vlsp2025")
pipeline_out = WORK_DIR / "data/pipeline"

# Use gold SFT data (NOT distilled) — this is the SFT-only baseline
train_path = data_paths.get("sft_train", str(pipeline_out / "sft_train.json"))
valid_path = data_paths.get("sft_valid", str(pipeline_out / "sft_valid.json"))

print("=" * 60)
print("SFT TRAINING (GOLD DATA, 12h Kaggle fit)")
print("=" * 60)
print(f"  Train: {train_path}")
print(f"  Valid: {valid_path}")
print(f"  LoRA r={cfg.sft.lora_r}  seq_len={cfg.sft.max_seq_length}  epochs={cfg.sft.num_epochs}")
print(f"  Runtime cap: {cfg.sft.max_runtime_hours}h  | Sample cap: {cfg.sft.max_train_samples}")
print(f"  Mirror dir : {cfg.sft.mirror_save_dir}")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free_gb = torch.cuda.mem_get_info()[0] / 1024**3
    print(f"  GPU free: {free_gb:.1f} GB")

# Override output dir so it doesn't conflict with the KD notebook.
# dataclasses.replace preserves all other fields including the 12h-fit ones.
cfg.sft = dataclasses.replace(cfg.sft, output_dir="checkpoints/sft_only")

# Resume from latest in-session checkpoint OR from the mirror if the prior
# session was killed (mirror survives kernel restarts; in-session dir may not).
def _latest_ckpt(root: Path):
    if not root.exists():
        return None
    ckpts = sorted(
        [d for d in root.iterdir() if d.is_dir() and d.name.startswith("checkpoint-")],
        key=lambda d: int(d.name.split("-")[-1]) if d.name.split("-")[-1].isdigit() else 0,
    )
    return str(ckpts[-1]) if ckpts else None

sft_ckpt_dir = WORK_DIR / cfg.sft.output_dir
resume_ckpt = _latest_ckpt(sft_ckpt_dir)
if resume_ckpt is None and cfg.sft.mirror_save_dir:
    mirror_latest = _latest_ckpt(Path(cfg.sft.mirror_save_dir))
    if mirror_latest:
        # Copy mirrored checkpoint into working dir so Trainer can resume from it.
        import shutil
        sft_ckpt_dir.mkdir(parents=True, exist_ok=True)
        dst = sft_ckpt_dir / Path(mirror_latest).name
        if not dst.exists():
            shutil.copytree(mirror_latest, dst)
        resume_ckpt = str(dst)
        print(f"  Resuming from MIRROR: {resume_ckpt}")
elif resume_ckpt:
    print(f"  Resuming from: {resume_ckpt}")
else:
    print("  Fresh run (no checkpoint found)")

t0 = time.time()
sft_model_path = run_sft_training(
    cfg,
    train_path=train_path,
    valid_path=valid_path,
    resume_from_checkpoint=resume_ckpt,
)
elapsed = time.time() - t0
print(f"\nSFT done in {elapsed/3600:.2f}h ({elapsed:.0f}s)")
print(f"Model: {sft_model_path}")

# Persist a canonical copy into OUTPUT_DIR for downstream cells / dataset upload.
import shutil
sft_out = Path(OUTPUT_DIR) / "sft_adapter"
if sft_out.exists():
    shutil.rmtree(sft_out)
try:
    shutil.copytree(sft_model_path, sft_out)
    _mb = sum(p.stat().st_size for p in sft_out.rglob("*") if p.is_file()) / 1024**2
    print(f"Adapter persisted: {sft_out} ({_mb:.0f} MB)")
except Exception as _e:
    print(f"WARN: adapter persist failed: {_e}")

gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()


SFT TRAINING (GOLD DATA)
  Train: /kaggle/working/vlsp2025/data/pipeline/sft_train.json
  Valid: /kaggle/working/vlsp2025/data/pipeline/sft_valid.json
  LoRA r=64  seq_len=2048


  GPU free: 94.4 GB


SFT train data: /kaggle/working/vlsp2025/data/pipeline/sft_train.json
SFT valid data: /kaggle/working/vlsp2025/data/pipeline/sft_valid.json


Train samples: 14661, Valid samples: 584
Loading student model: /kaggle/input/models/thanhduc1102/qwen_35_4b/transformers/default/1
[GPU-MEM before-student-load] free=94.4GB total=95.0GB alloc=0.0GB reserved=0.0GB


`torch_dtype` is deprecated! Use `dtype` instead!


...

  [SFT step 2150/2748]  loss=0.0295  grad_norm=0.251  elapsed=684.9min  ETA=191min


  [SFT step 2175/2748]  loss=0.0244  grad_norm=2.544  elapsed=692.4min  ETA=182min


  [SFT step 2200/2748]  loss=0.0183  grad_norm=1.157  elapsed=699.9min  ETA=174min


  [SFT step 2225/2748]  loss=0.0159  grad_norm=3.071  elapsed=709.3min  ETA=167min


  [SFT step 2250/2748]  loss=0.0171  grad_norm=1.266  elapsed=716.8min  ETA=159min


In [ ]:
# ═════════════════════════════════════════════════════════════════════
# GREEDY EVALUATION — fast single-pass batched inference
# Time: ~40-60 min for 584 valid samples with Qwen3.5-4B + RTX 6000
# vs 35h for majority voting (num_candidates=15)
# ═════════════════════════════════════════════════════════════════════
import gc, json, time, torch, warnings
from pathlib import Path
from tqdm import tqdm
from pipeline import _load_model_robust, _load_tokenizer_robust
from pipeline.evaluate import answers_match, programs_match
from pipeline.program_executor import validate_program
import re

# Suppress non-critical deprecation warnings (torch_dtype, warmup_ratio)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, message=".*warmup_ratio.*")

WORK_DIR = Path("/kaggle/working/vlsp2025")

VALID_PATH = globals().get("data_paths", {}).get("sft_valid") or str(WORK_DIR / "data/pipeline/sft_valid.json")
with open(VALID_PATH, "r", encoding="utf-8") as f:
    valid_data = json.load(f)

if TEST_MODE:
    valid_data = valid_data[:min(TEST_SAMPLES, len(valid_data))]
    print(f"TEST MODE: using {len(valid_data)} validation samples")

# Resolve display name for the model (avoid showing "1" from .../default/1)
_model_display = sft_model_path
for _part in reversed(sft_model_path.replace("\\", "/").split("/")):
    if len(_part) > 3 and _part not in ("final", "default", "1"):
        _model_display = _part; break
print(f"Evaluating {len(valid_data)} samples | Model: {_model_display}")

# --- Load model ---
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

dtype = torch.float16 if cfg.model.torch_dtype == "float16" else torch.bfloat16
# Support both old (torch_dtype) and new (dtype) transformers API
import inspect as _inspect
from transformers import AutoModelForCausalLM as _ACM
_dtype_key = "dtype" if "dtype" in _inspect.signature(_ACM.from_pretrained).parameters else "torch_dtype"
mkw = {"trust_remote_code": True, _dtype_key: dtype, "device_map": "auto", "low_cpu_mem_usage": True}
if cfg.model.use_flash_attention:
    try:
        import flash_attn
        mkw["attn_implementation"] = "flash_attention_2"
    except ImportError:
        mkw["attn_implementation"] = "sdpa"

tokenizer = _load_tokenizer_robust(sft_model_path, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

try:
    from peft import PeftModel
    base = _load_model_robust(cfg.model.student_model, mkw)
    model = PeftModel.from_pretrained(base, sft_model_path)
    print("Loaded as PEFT (LoRA) model")
except Exception as _e:
    print(f"PEFT load skipped ({type(_e).__name__}), loading as full model")
    model = _load_model_robust(sft_model_path, mkw)

model.eval()
model.config.pad_token_id = tokenizer.pad_token_id
free_gb = torch.cuda.mem_get_info()[0] / 1024**3
print(f"Model loaded. GPU free: {free_gb:.1f} GB")

# --- Helper: extract program/answer from output text ---
def _extract(text):
    pm = list(re.finditer(r"\*\*Chương trình tính toán:\*\*\s*((?:.|\n)*?)(?=\s*\*\*|$)", text))
    am = list(re.finditer(r"\*\*Đáp án cuối cùng:\*\*\s*((?:.|\n)*?)(?=\s*\*\*|$)", text))
    prog = pm[-1].group(1).strip() if pm else None
    ans  = am[-1].group(1).strip() if am else None
    return prog, ans

# --- Batched greedy inference ---
BATCH_SIZE = 8
MAX_NEW_TOKENS = 512  # output format ~100-300 tokens; 512 is safe ceiling

eval_results = []
batches = [valid_data[i:i+BATCH_SIZE] for i in range(0, len(valid_data), BATCH_SIZE)]
t0 = time.time()

for b_idx, batch in enumerate(tqdm(batches, desc="Greedy eval")):
    prompts  = [s["messages"][0]["content"] for s in batch]
    gold_prog = [s.get("metadata", {}).get("program", "") for s in batch]
    gold_ans  = [str(s.get("metadata", {}).get("answer", "")) for s in batch]

    # Chat template
    texts_in = []
    for p in prompts:
        msgs = [{"role": "user", "content": p}]
        try:
            t = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
        except TypeError:
            t = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        texts_in.append(t)

    enc = tokenizer(texts_in, return_tensors="pt", truncation=True, max_length=3584, padding=True)
    enc = {k: v.to(model.device) for k, v in enc.items()}
    in_len = enc["input_ids"].shape[1]

    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS,
                             do_sample=False, pad_token_id=tokenizer.pad_token_id)
    gen = tokenizer.batch_decode(out[:, in_len:], skip_special_tokens=True)

    for i, (txt, samp) in enumerate(zip(gen, batch)):
        pp, pa = _extract(txt)
        ea = answers_match(pa or "", gold_ans[i])
        pa_m = programs_match(pp or "", gold_prog[i])
        eval_results.append({
            "id": samp.get("id", f"s{len(eval_results)}"),
            "pred_program": pp or "",
            "pred_answer":  pa or "",
            "gold_program": gold_prog[i],
            "gold_answer":  gold_ans[i],
            "ea": ea, "pa": pa_m,
            "valid": validate_program(pp) if pp else False,
            "raw_output": txt[:300],
        })

    # Progress log every 10 batches
    if (b_idx + 1) % 10 == 0:
        el = time.time() - t0
        eta = (len(batches) - b_idx - 1) / ((b_idx + 1) / el)
        n_done = min((b_idx + 1) * BATCH_SIZE, len(valid_data))
        ea_now = sum(r["ea"] for r in eval_results) / len(eval_results) * 100
        print(f"  [{n_done}/{len(valid_data)}] EA={ea_now:.1f}%  ETA={eta/60:.0f}min")

elapsed = time.time() - t0
N = len(eval_results)
EA = sum(r["ea"] for r in eval_results) / N * 100 if N else 0
PA = sum(r["pa"] for r in eval_results) / N * 100 if N else 0
VR = sum(r["valid"] for r in eval_results) / N * 100 if N else 0

print(f"\n{'='*60}")
print(f"GREEDY EVALUATION ({N} samples,  {elapsed:.0f}s / {elapsed/60:.1f}min)")
print(f"{'='*60}")
print(f"  Execution Accuracy (EA): {EA:.2f}%   ({sum(r['ea'] for r in eval_results)}/{N})")
print(f"  Program Accuracy   (PA): {PA:.2f}%   ({sum(r['pa'] for r in eval_results)}/{N})")
print(f"  Valid Program Rate:      {VR:.2f}%")
print(f"{'='*60}")

eval_summary = {
    "execution_accuracy": EA / 100,
    "program_accuracy":   PA / 100,
    "valid_rate":         VR / 100,
    "total": N,
    "ea_correct": sum(r["ea"] for r in eval_results),
    "pa_correct": sum(r["pa"] for r in eval_results),
    "details": eval_results,
}

# Free GPU after eval — model no longer needed
del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free_gb = torch.cuda.mem_get_info()[0] / 1024**3
    print(f"GPU free after eval cleanup: {free_gb:.1f} GB")


In [ ]:
# ═════════════════════════════════════════════════════════════════════
# DETAILED ERROR ANALYSIS — breakdown of prediction quality
# ═════════════════════════════════════════════════════════════════════
import json
from pathlib import Path

N = len(eval_results)
perfect   = sum(1 for r in eval_results if r["ea"] and r["pa"])
ea_only   = sum(1 for r in eval_results if r["ea"] and not r["pa"])
valid_err = sum(1 for r in eval_results if not r["ea"] and r["valid"])
inv_prog  = sum(1 for r in eval_results if not r["ea"] and not r["valid"] and r["pred_program"])
no_prog   = sum(1 for r in eval_results if not r["pred_program"])

print(f"\n{'='*60}")
print(f"ERROR ANALYSIS  (N={N})")
print(f"{'='*60}")
print(f"{'Category':<35} {'Count':>6}  {'%':>6}")
print(f"{'-'*50}")
print(f"{'EA + PA correct (perfect)' :<35} {perfect:>6}  {perfect/N*100:>5.1f}%")
print(f"{'EA correct, PA wrong (alt solution)':<35} {ea_only:>6}  {ea_only/N*100:>5.1f}%")
print(f"{'EA wrong, valid program':<35} {valid_err:>6}  {valid_err/N*100:>5.1f}%")
print(f"{'EA wrong, invalid program':<35} {inv_prog:>6}  {inv_prog/N*100:>5.1f}%")
print(f"{'No program generated':<35} {no_prog:>6}  {no_prog/N*100:>5.1f}%")
print(f"{'-'*50}")
print(f"{'Total':<35} {N:>6}  100.0%")
print(f"{'='*60}")

# Show 5 failure examples
print("\nFAILURE EXAMPLES (EA wrong):")
failures = [r for r in eval_results if not r["ea"]][:5]
for i, r in enumerate(failures, 1):
    print(f"\n[{i}] id={r['id']}")
    print(f"  Gold  : program={r['gold_program'][:60]}  answer={r['gold_answer']}")
    print(f"  Pred  : program={r['pred_program'][:60]}  answer={r['pred_answer']}")
    print(f"  Valid : {r['valid']}")


In [ ]:
import gc, json, shutil, torch
from pathlib import Path

WORK_DIR = Path("/kaggle/working/vlsp2025")
out = Path(OUTPUT_DIR)
out.mkdir(parents=True, exist_ok=True)
pipeline_out = WORK_DIR / "data/pipeline"

# Build and save eval_results.json
eval_out = dict(notebook=NOTEBOOK_ID, test_mode=TEST_MODE, model_path=sft_model_path)
for k, v in eval_summary.items():
    if k != "details":
        eval_out[k] = v
eval_out["details"] = eval_summary["details"]
with open(out / "eval_results.json", "w", encoding="utf-8") as f:
    json.dump(eval_out, f, ensure_ascii=False, indent=2)
print(f"eval_results.json saved  (EA={eval_out['execution_accuracy']:.2%}  PA={eval_out['program_accuracy']:.2%})")

# Compact predictions for cross-notebook comparison
preds_compact = [
    dict(id=r["id"], pred_answer=r["pred_answer"], gold_answer=r["gold_answer"],
         ea=r["ea"], pa=r["pa"])
    for r in eval_summary["details"]
]
with open(out / "predictions.json", "w", encoding="utf-8") as f:
    json.dump(preds_compact, f, ensure_ascii=False, indent=2)

# Save model adapter (LoRA weights)
model_save = out / "sft_adapter"
if model_save.exists():
    shutil.rmtree(model_save)
if Path(sft_model_path).exists():
    shutil.copytree(sft_model_path, model_save)
    sz = sum(p.stat().st_size for p in model_save.rglob("*") if p.is_file()) / 1024**2
    print(f"Model adapter saved -> {model_save}  ({sz:.0f} MB)")
else:
    print(f"WARNING: sft_model_path not found: {sft_model_path}")

# Copy data / config files
for fname in ["sft_train.json", "sft_valid.json", "config_sft_only.yaml"]:
    src = pipeline_out / fname
    if src.exists():
        shutil.copy2(src, out / fname)

print(f"\nAll outputs -> {OUTPUT_DIR}")
print(f"Files: {sorted(p.name for p in out.iterdir())}")
print(f"\nSUMMARY  ({'TEST MODE - ' + str(TEST_SAMPLES) + ' samples' if TEST_MODE else 'FULL RUN'}):")
print(f"  EA (Execution Accuracy): {eval_summary['execution_accuracy']:.2%}")
print(f"  PA (Program Accuracy)  : {eval_summary['program_accuracy']:.2%}")
print(f"  Valid Program Rate     : {eval_summary['valid_rate']:.2%}")
if TEST_MODE:
    print(f"\n  ⚠ TEST_MODE=True — numbers above are meaningless (only {TEST_SAMPLES} training samples).")
    print(f"  ⚠ Set TEST_MODE=False, restart kernel, and rerun for real results.")
